Garrett Whiteker
Advanced Data and Big Data Mining
Project Deliverable 3

 Deliverable 3: Classification, Clustering, and Association Rule Mining

This project focuses on applying advanced data mining techniques to analyze health indicators and diabetes risk. The analysis includes classification models, clustering techniques, and association rule mining.

The objectives of this deliverable are:
- To build and evaluate classification models for predicting diabetes
- To improve model performance using hyperparameter tuning
- To identify natural groupings in the data using clustering
- To discover relationships between variables using association rule mining

These techniques provide deeper insights into patterns within the dataset and help support data-driven decision-making in real-world applications.

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, roc_curve, auc
from sklearn.preprocessing import StandardScaler

from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.cluster import KMeans

import matplotlib.pyplot as plt
import seaborn as sns

from mlxtend.frequent_patterns import apriori, association_rules

Data Loading and Preparation

The dataset is loaded into a Pandas DataFrame for analysis. A binary target variable is created to simplify the classification task, where individuals are labeled as either having diabetes or not.

This transformation allows the use of classification algorithms to predict diabetes risk based on health and demographic features.

In [ ]:
df = pd.read_csv("diabetes_012_health_indicators_BRFSS2015.csv")

# Create binary target
df['Diabetes_binary'] = (df['Diabetes_012'] > 0).astype(int)

df.head()

Train-Test Split

The dataset is divided into training and testing sets. The training set is used to build the models, while the testing set is used to evaluate their performance on unseen data.

An 80/20 split is used to ensure that sufficient data is available for both training and evaluation.

In [ ]:
X = df.drop(columns=["Diabetes_012", "Diabetes_binary"])
y = df["Diabetes_binary"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

Classification Models

Two classification models are developed to predict diabetes status:

- Decision Tree: A model that splits data based on feature values to make predictions.
- k-Nearest Neighbors (k-NN): A model that classifies data points based on similarity to neighboring points.

These models are chosen because they represent different approaches to classification and allow for performance comparison.

In [ ]:
dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)

Model Evaluation

The performance of each model is evaluated using the following metrics:

- Accuracy: Measures the proportion of correct predictions.
- F1 Score: Balances precision and recall, making it useful for imbalanced datasets.

These metrics provide insight into how well each model predicts diabetes status.

In [ ]:
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)
y_pred_knn = knn.predict(X_test)

 Confusion Matrix Analysis

The confusion matrix provides a detailed breakdown of prediction outcomes, including:

- True Positives
- True Negatives
- False Positives
- False Negatives

This helps evaluate how well the model distinguishes between individuals with and without diabetes.

In [ ]:
def evaluate(y_true, y_pred, name):
    print(f"\n{name}")
    print("Accuracy:", accuracy_score(y_true, y_pred))
    print("F1 Score:", f1_score(y_true, y_pred))

evaluate(y_test, y_pred_dt, "Decision Tree")
evaluate(y_test, y_pred_knn, "k-NN")

ROC Curve Analysis

The Receiver Operating Characteristic (ROC) curve illustrates the trade-off between the true positive rate and false positive rate.

The Area Under the Curve (AUC) indicates the model's ability to distinguish between classes. A higher AUC value represents better model performance.

In [ ]:
# Decision Tree
cm_dt = confusion_matrix(y_test, y_pred_dt)
sns.heatmap(cm_dt, annot=True, fmt="d")
plt.title("Decision Tree Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

# k-NN
cm_knn = confusion_matrix(y_test, y_pred_knn)
sns.heatmap(cm_knn, annot=True, fmt="d")
plt.title("k-NN Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

 Hyperparameter Tuning

Hyperparameter tuning is performed using GridSearchCV to improve model performance. Different combinations of parameters such as maximum tree depth and minimum samples per split are tested.

The best parameter combination is selected based on cross-validation results, helping to reduce overfitting and improve generalization.

In [ ]:
# Decision Tree
y_probs_dt = dt.predict_proba(X_test)[:,1]
fpr_dt, tpr_dt, _ = roc_curve(y_test, y_probs_dt)

# k-NN
y_probs_knn = knn.predict_proba(X_test)[:,1]
fpr_knn, tpr_knn, _ = roc_curve(y_test, y_probs_knn)

plt.plot(fpr_dt, tpr_dt, label="Decision Tree")
plt.plot(fpr_knn, tpr_knn, label="k-NN")
plt.plot([0,1], [0,1], linestyle='--')

plt.legend()
plt.title("ROC Curve Comparison")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.show()

 Model Comparison

The performance of the classification models is compared using accuracy scores. This comparison helps determine which model performs better for predicting diabetes.

The results highlight differences in model effectiveness and guide the selection of the best-performing model.

In [ ]:
param_grid = {
    'max_depth': [3, 5, 10],
    'min_samples_split': [2, 5, 10]
}

grid = GridSearchCV(DecisionTreeClassifier(), param_grid, cv=5)
grid.fit(X_train, y_train)

print("Best Parameters:", grid.best_params_)

Clustering Analysis

K-Means clustering is applied to group individuals based on similarities in their health indicators.

Clustering helps uncover hidden patterns in the data and identifies groups of individuals with similar characteristics, which may correspond to different levels of health risk.

In [ ]:
models = ["Decision Tree", "k-NN"]
accuracy = [
    accuracy_score(y_test, y_pred_dt),
    accuracy_score(y_test, y_pred_knn)
]

plt.bar(models, accuracy)
plt.title("Model Accuracy Comparison")
plt.ylabel("Accuracy")
plt.show()

 Cluster Visualization

The clusters are visualized using BMI and Age as key features. Each point represents an individual, and colors indicate cluster membership.

This visualization helps interpret how individuals are grouped and highlights differences between clusters.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

kmeans = KMeans(n_clusters=3, random_state=42)
df['Cluster'] = kmeans.fit_predict(X_scaled)

Association Rule Mining

Association rule mining is used to discover relationships between health indicators.

The Apriori algorithm identifies frequent itemsets and generates rules based on support and confidence metrics. These rules reveal patterns in how different health conditions are related.

In [ ]:
plt.scatter(df['BMI'], df['Age'], c=df['Cluster'])
plt.xlabel("BMI")
plt.ylabel("Age")
plt.title("K-Means Clusters")
plt.show()

Association Rule Insights

The discovered rules indicate that combinations of health conditions, such as high blood pressure and high cholesterol, are associated with an increased likelihood of diabetes.

These patterns suggest that multiple risk factors together significantly influence health outcomes.

Such insights can be applied in healthcare to identify high-risk individuals and support preventative care strategies.

In [ ]:
df_rules = df[['HighBP','HighChol','Smoker','PhysActivity','Diabetes_binary']]

frequent_itemsets = apriori(df_rules, min_support=0.1, use_colnames=True)

rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.6)

rules.sort_values(by="confidence", ascending=False).head()

Final Insights and Conclusion

In this analysis, multiple data mining techniques were used to better understand patterns related to diabetes risk.

The classification models showed that the Decision Tree performed slightly better overall, especially after tuning its parameters. This suggests that it was more effective at capturing relationships in the data compared to the k-NN model.

Clustering helped group individuals with similar health characteristics, revealing that people can be segmented into different health profiles. Some groups appear to have higher risk factors, which could be useful for targeting preventative care.

The association rule mining step uncovered patterns between conditions like high blood pressure, high cholesterol, and diabetes. These results highlight that diabetes risk is often influenced by a combination of factors rather than a single variable.

Overall, this project shows that using multiple techniques together provides a clearer and more complete understanding of the data. These insights could be useful in real-world healthcare settings to identify at-risk individuals and support early intervention strategies.